# 05 — Recovering `tx` from grains (the geometry powder cannot see)

A powder / calibrant fit (`midas-calibrate-v2`) pins `Lsd`, the beam centre,
`ty`, `tz` and the distortion — but it is **structurally blind to `tx`**, the
in-plane detector rotation about the beam. Debye–Scherrer rings are
rotationally symmetric about the beam axis, so rotating the detector in its
own plane maps every ring onto itself. No powder pattern, however good, carries
information about `tx`.

`tx` can only be pinned by **single-crystal spots**, whose azimuthal positions
break that symmetry. That is what this package's `grain-tx` step does, and it
is why an FF reconstruction is properly a **two-pass** procedure:

1. calibrate on the calibrant → reconstruct with `tx = 0`;
2. refine `tx` from the recovered grains → **re-reconstruct**.

Leaving `tx` at 0 when it is not 0 shifts every spot by `R·sin(tx)` — for a
real Ni dataset, `tx = 0.049°` at `R ≈ 150 mm` is a ~130 µm azimuthal shift,
which corrupts indexing *and* the refined positions and strains.

### What this notebook covers

| Part | Data | Runs in CI |
| --- | --- | --- |
| 1. `tx` moves η, not R | synthetic | yes |
| 2. Why the grain pose must be frozen | synthetic | yes |
| 3. Recovering a known `tx` from 0 | synthetic | yes |
| 4. The production path on a real reconstruction | bring-your-own | no |

Parts 1–3 are self-contained — no dataset, no subprocess, no network. They use
the same generators as `tests/test_grain_refine.py`.


In [1]:
import os, math
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')   # macOS OpenMP guard
import numpy as np
import torch

import midas_peakfit as mp
from midas_peakfit import Parameter
from midas_calibrate.geometry_torch import build_tilt_matrix_torch, pixel_to_REta_torch
from midas_diffract import HEDMForwardModel
from midas_diffract.forward import HEDMGeometry
from midas_diffract.hkls import hkls_for_forward_model
from midas_hkls import Lattice, SpaceGroup
from midas_fit_grain.matching import MatchResult
from midas_fit_grain.observations import ObservedSpots
from midas_joint_ff_calibrate.grain_refine import make_residual
from midas_joint_ff_calibrate.spec import build_joint_spec
from midas_stress.orientation import euler_to_orient_mat, orient_mat_to_euler

torch.manual_seed(0)
DT = torch.float64

# One synthetic FF geometry: Ni FCC, Varex-like 2048 x 2048 @ 150 um, 1 m.
LSD, BCY, BCZ, PX = 1.0e6, 1024.0, 1024.0, 150.0
RHOD, NPIX = 1024.0 * PX, 2048


def build_model():
    sg = SpaceGroup.from_number(225)                       # Ni, Fm-3m
    lat = Lattice(3.6, 3.6, 3.6, 90.0, 90.0, 90.0)
    hkls_cart, thetas, hkls_int = hkls_for_forward_model(
        sg, lat, wavelength_A=0.2066, two_theta_max_deg=18.0,
        expand_equivalents=True)
    geom = HEDMGeometry(
        Lsd=LSD, y_BC=BCY, z_BC=BCZ, px=PX, omega_start=-180.0, omega_step=0.25,
        n_frames=1440, n_pixels_y=NPIX, n_pixels_z=NPIX, min_eta=6.0,
        wavelength=0.2066, tx=0.0, ty=0.0, tz=0.0, wedge=0.0,
        flip_y=True, apply_tilts=False, multi_mode='layered')
    return HEDMForwardModel(hkls_cart, thetas, geom, hkls_int=hkls_int.float())


def bake_tx_into_raw(yp_ideal, zp_ideal, tx_true_deg):
    """RAW pixels such that DetCor(raw, tx_true) == the ideal tilt-free pixels.

    With ty=tz=0 and no distortion the pure-tx tilt is an exact 2-D rotation
    of the centred (Yc, Zc) detector coordinates, so we simply apply its
    inverse."""
    T = build_tilt_matrix_torch(torch.tensor(tx_true_deg, dtype=DT),
                                torch.tensor(0.0, dtype=DT),
                                torch.tensor(0.0, dtype=DT))
    Rot = T[1:, 1:]                                # 2x2 rotation on (Yc, Zc)
    Yc = (-yp_ideal + BCY) * PX
    Zc = (zp_ideal - BCZ) * PX
    raw = torch.stack([Yc, Zc], dim=-1) @ Rot      # orthonormal => == Rot^T . v
    return BCY - raw[..., 0] / PX, BCZ + raw[..., 1] / PX


def build_synth(tx_true_deg, n_grains=8, seed=0, max_spots=40):
    """Forward-model grains in ideal space, bake tx_true into the RAW pixels,
    and assemble (observations, matches, raw_yz) with an identity match."""
    model = build_model()
    rng = np.random.default_rng(seed)
    eulers = rng.uniform(-math.pi, math.pi, size=(n_grains, 3))
    eulers[:, 1] = rng.uniform(0, math.pi, size=n_grains)      # Phi in [0, pi]
    positions = np.zeros((n_grains, 3))
    lattices = np.tile(np.array([3.6, 3.6, 3.6, 90.0, 90.0, 90.0]), (n_grains, 1))

    observations, matches, raw_yz = [], [], []
    eul_t = torch.from_numpy(eulers).to(DT)
    pos_t = torch.from_numpy(positions).to(DT)
    lat_t = torch.from_numpy(lattices).to(DT)

    def sq(t):
        while t.dim() > 2 and t.shape[0] == 1:
            t = t.squeeze(0)
        return t

    for g in range(n_grains):
        s = model(eul_t[g].view(1, 1, 3), pos_t[g].view(1, 1, 3),
                  lattice_params=lat_t[g].view(1, 6))
        valid = sq(s.valid).bool()
        yp, zp = sq(s.y_pixel).double(), sq(s.z_pixel).double()
        om, eta, tth = sq(s.omega).double(), sq(s.eta).double(), sq(s.two_theta).double()
        M = valid.shape[1]
        ks, msq = torch.where(valid)
        if ks.numel() == 0:
            continue
        if ks.numel() > max_spots:
            sel = torch.randperm(ks.numel())[:max_spots]
            ks, msq = ks[sel], msq[sel]
        flat = ks * M + msq
        yp_i, zp_i = yp.reshape(-1)[flat], zp.reshape(-1)[flat]
        om_i = om.reshape(-1)[flat]
        eta_i, tth_i = eta.reshape(-1)[flat], tth.reshape(-1)[flat]
        yp_raw, zp_raw = bake_tx_into_raw(yp_i, zp_i, tx_true_deg)

        # Observed lab (Y, Z): the ideal prediction R=Lsd*tan(2theta),
        # (Y,Z)=(-R sin eta, R cos eta), rotated about the beam by -tx_true.
        # make_residual rotates the OBSERVED by the trial tx and matches it to
        # the ideal prediction, so the (Y,Z) minimum sits exactly at tx_true.
        R_id = LSD * torch.tan(tth_i)
        pred_vec = torch.stack([-R_id * torch.sin(eta_i),
                                R_id * torch.cos(eta_i)], dim=-1)
        T_tx = build_tilt_matrix_torch(torch.tensor(tx_true_deg, dtype=DT),
                                       torch.tensor(0.0, dtype=DT),
                                       torch.tensor(0.0, dtype=DT))
        obs_vec = pred_vec @ T_tx[1:, 1:]
        S = ks.numel()
        observations.append(ObservedSpots(
            spot_id=torch.arange(S), ring_nr=torch.zeros(S, dtype=torch.int64),
            y_lab=obs_vec[..., 0], z_lab=obs_vec[..., 1],
            omega=om_i, eta=eta_i, two_theta=tth_i,
            grain_radius=torch.full((S,), 50.0, dtype=DT),
            fit_rmse=torch.zeros(S, dtype=DT), y_orig=torch.zeros(S, dtype=DT),
            z_orig=torch.zeros(S, dtype=DT), omega_ini=om_i.clone(),
            mask_touched=torch.zeros(S, dtype=torch.bool)))
        matches.append(MatchResult(
            k_idx=ks.long(), m_idx=msq.long(), mask=torch.ones(S, dtype=torch.bool),
            delta_omega=torch.zeros(S, dtype=DT), delta_eta=torch.zeros(S, dtype=DT)))
        raw_yz.append((yp_raw, zp_raw))
    return model, observations, matches, raw_yz, eulers, positions, lattices


def fixed_geo():
    return dict(
        Lsd=torch.tensor(LSD, dtype=DT), BC_y=torch.tensor(BCY, dtype=DT),
        BC_z=torch.tensor(BCZ, dtype=DT), ty=torch.tensor(0.0, dtype=DT),
        tz=torch.tensor(0.0, dtype=DT), px=torch.tensor(PX, dtype=DT),
        RhoD=torch.tensor(RHOD, dtype=DT), p_coeffs=torch.zeros(15, dtype=DT))


def make_spec(eulers, positions, lattices, tx_init=0.0, free_pose=False):
    """tx + the three grain blocks. free_pose=True thaws grain orientation --
    used ONLY as the negative control in part 2."""
    spec = mp.ParameterSpec()
    spec.add(Parameter('tx', init=torch.tensor(tx_init, dtype=DT), refined=True,
                       bounds=(-5.0, 5.0)))
    spec = build_joint_spec(
        powder_spec=spec,
        grain_eulers_init=torch.from_numpy(eulers).to(DT),
        grain_positions_init=torch.from_numpy(positions).to(DT),
        grain_lattices_init=torch.from_numpy(lattices).to(DT),
        refine_grain_orientation=free_pose, refine_grain_position=False,
        refine_grain_strain=False)
    if free_pose:
        spec.parameters['grain_euler'].bounds = (-2 * math.pi, 2 * math.pi)
    return spec


def cost_at(spec, resid, **overrides):
    u = {n: spec.parameters[n].init_tensor() for n in spec.parameters}
    for k, v in overrides.items():
        u[k] = torch.tensor(np.asarray(v), dtype=DT)
    return float((resid(u) ** 2).sum())


def rotate_grains_about_beam(eul, delta_deg):
    """Rotate every grain orientation about the LAB beam axis (X) by delta.

    This is the operation that -- if the tx <-> orientation degeneracy were
    exact -- would let the grains absorb a tx error. Part 2 measures whether
    it actually does."""
    d = math.radians(delta_deg)
    Rx = np.array([[1.0, 0.0, 0.0],
                   [0.0, math.cos(d), -math.sin(d)],
                   [0.0, math.sin(d), math.cos(d)]])
    out = np.zeros_like(eul)
    for i, e in enumerate(eul):
        OM = np.asarray(euler_to_orient_mat(e)).reshape(3, 3)
        out[i] = orient_mat_to_euler((Rx @ OM).reshape(9))
    return out


print('preamble ready')


DIPlib -- a quantitative image analysis library
Version 3.5.2 (Dec 27 2024)
For more information see https://diplib.org
preamble ready


/Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. `tx` rotates η and leaves R alone

Take spots whose RAW pixels contain a known `tx`, then run the DetCor
transform (`pixel_to_REta_torch`) twice — once with the correct `tx`, once with
`tx = 0` — and compare the recovered radius `R` and azimuth `η`.

This measurement is the whole reason `kind="pixel"` is **disabled** for the
`grain-tx` step: a radial/pixel-distance residual barely moves with `tx`, so it
cannot constrain it. The loss has to be η-sensitive.


In [2]:
TX_TRUE = 0.40                      # degrees
model, obs, matches, raw_yz, eulers, positions, lattices = build_synth(TX_TRUE)
fg = fixed_geo()
print(f'{len(obs)} grains, {sum(int(m.mask.sum()) for m in matches)} spots')

Yr, Zr = raw_yz[0]
def detcor(tx_deg):
    return pixel_to_REta_torch(
        Yr, Zr, Lsd=fg['Lsd'], BC_y=fg['BC_y'], BC_z=fg['BC_z'],
        tx=torch.tensor(tx_deg, dtype=DT), ty=fg['ty'], tz=fg['tz'],
        p_coeffs=fg['p_coeffs'], parallax=torch.zeros((), dtype=DT),
        px=fg['px'], rho_d=fg['RhoD'])

print(f'\n{"tx used":>9s}  {"tx error":>10s}  {"max |dR|":>12s}  {"max |d.eta|":>12s}')
print(f'{"(deg)":>9s}  {"(deg)":>10s}  {"(um)":>12s}  {"(deg)":>12s}')
R_ref, eta_ref = detcor(TX_TRUE)
for tx_try in (0.0, 0.1, 0.2, 0.4):
    R_t, eta_t = detcor(tx_try)
    dR = float((R_ref - R_t).abs().max()) * float(fg['px'])
    deta = float((eta_ref - eta_t).abs().max())
    print(f'{tx_try:>9.2f}  {TX_TRUE - tx_try:>10.2f}  {dR:>12.6f}  {deta:>12.4f}')
print('\n=> d.eta tracks the tx error 1:1; dR is zero to numerical precision.')
print('   A radial (pixel) loss is blind to tx; an angular loss is not.')


8 grains, 320 spots

  tx used    tx error      max |dR|   max |d.eta|
    (deg)       (deg)          (um)         (deg)
     0.00        0.40      0.000000        0.4000
     0.10        0.30      0.000000        0.3000
     0.20        0.20      0.000000        0.2000
     0.40        0.00      0.000000        0.0000

=> d.eta tracks the tx error 1:1; dR is zero to numerical precision.
   A radial (pixel) loss is blind to tx; an angular loss is not.


## 2. Why the grain pose must be frozen

`tx` rotates the **observed** pattern about the beam. A grain's orientation
rotates the **predicted** pattern about the beam. That looks like a textbook
degeneracy — counter-rotate every grain and you should be able to absorb any
`tx` — and it is the stated reason production `refine_geometry_from_grains`
**holds grain orientation, position and strain fixed** for the `tx` step.

It is worth checking rather than repeating, because it turns out to be only
half true. Two measurements below:

1. a **direct probe** — offset `tx`, counter-rotate every grain about the beam
   by the same angle, and see whether the cost comes back;
2. the **operational test** — thaw the grain orientations and refine `tx`
   alongside them.

First, the frozen-pose cost scan, as a baseline.


In [3]:
spec_fixed = make_spec(eulers, positions, lattices, tx_init=0.0)
resid = make_residual(model, obs, matches, raw_yz, fixed_geo=fg, kind='angular')

scan = np.linspace(-0.6, 0.6, 49)
costs = np.array([cost_at(spec_fixed, resid, tx=float(t)) for t in scan])
print(f'cost minimum at tx = {scan[costs.argmin()]:+.3f} deg  (true {TX_TRUE:+.3f})')
print(f'cost at tx=0      : {cost_at(spec_fixed, resid, tx=0.0):.4e}')
print(f'cost at tx=tx_true: {cost_at(spec_fixed, resid, tx=TX_TRUE):.4e}')


cost minimum at tx = +0.400 deg  (true +0.400)
cost at tx=0      : 3.2671e+08
cost at tx=tx_true: 1.2500e-19


In [4]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5.6, 4.0))
ax.semilogy(scan, costs, 'b.-', lw=1.2, ms=4)
ax.axvline(TX_TRUE, color='g', ls='--', lw=1.5, label=f'true tx = {TX_TRUE}')
ax.axvline(0.0, color='r', ls=':', lw=1.5, label='powder pass (tx = 0)')
ax.set_xlabel('trial tx (deg)'); ax.set_ylabel('sum of squared residuals')
ax.set_title('Frozen-pose cost has a clean minimum at the true tx')
ax.legend(fontsize=8); fig.tight_layout()
fig.savefig('grain_tx_cost_scan.png', dpi=120)
plt.close(fig)
print('wrote grain_tx_cost_scan.png')


wrote grain_tx_cost_scan.png


### Probe 1 — can counter-rotating the grains actually absorb a `tx` error?

Offset `tx` by `delta`, rotate every grain about the lab beam axis by the same
`delta`, and compare against the *unrotated* cost at that same offset `tx`. If
the degeneracy were exact the counter-rotated cost would return to the floor.
The unrotated column is the control that makes the comparison meaningful.


In [5]:
print(f'{"delta":>7s}  {"cost @ tx+delta":>17s}  {"...+ grains rotated":>21s}  {"verdict":>12s}')
print(f'{"(deg)":>7s}  {"(no rotation)":>17s}  {"(counter-rotated)":>21s}  {"":>12s}')
for delta in (0.05, 0.10, 0.20):
    c_plain = cost_at(spec_fixed, resid, tx=TX_TRUE + delta)
    c_rot = cost_at(spec_fixed, resid, tx=TX_TRUE + delta,
                    grain_euler=rotate_grains_about_beam(eulers, delta))
    verdict = 'absorbed' if c_rot < 0.1 * c_plain else 'NOT absorbed'
    print(f'{delta:>7.2f}  {c_plain:>17.4e}  {c_rot:>21.4e}  {verdict:>12s}')
print(f'\nfloor (tx exact, pose exact): {cost_at(spec_fixed, resid, tx=TX_TRUE):.4e}')
print('\n=> Counter-rotating the grains does NOT recover the cost - it makes it')
print('   WORSE. The degeneracy is not exact, because tx leaves omega untouched')
print('   while rotating a grain about the beam shifts the omega at which each')
print('   spot diffracts. The omega term in the 3-D loss is what breaks it.')


  delta    cost @ tx+delta    ...+ grains rotated       verdict
  (deg)      (no rotation)      (counter-rotated)              
   0.05         5.1049e+06             7.5043e+06  NOT absorbed
   0.10         2.0420e+07             3.0018e+07  NOT absorbed
   0.20         8.1678e+07             1.2007e+08  NOT absorbed

floor (tx exact, pose exact): 1.2500e-19

=> Counter-rotating the grains does NOT recover the cost - it makes it
   WORSE. The degeneracy is not exact, because tx leaves omega untouched
   while rotating a grain about the beam shifts the omega at which each
   spot diffracts. The omega term in the 3-D loss is what breaks it.


### Probe 2 — the operational test

The degeneracy is not exact, so in principle `tx` and the grain orientations are
jointly identifiable. In practice: thaw the orientations (24 extra parameters
for 8 grains) and refine `tx` alongside them from the same start.


In [6]:
def run_lm(spec, resid, max_iter=80):
    u0 = {n: spec.parameters[n].init_tensor() for n in spec.parameters}
    c0 = float((resid(u0) ** 2).sum())
    u, c, rc = mp.lm_minimise(
        spec, resid,
        config=mp.GenericLMConfig(max_iter=max_iter, ftol_rel=1e-12, xtol_rel=1e-12),
        fallback_span=2.0)
    return float(u['tx']), c0, float(c), rc

tx_fixed, c0_f, c1_f, rc_f = run_lm(
    make_spec(eulers, positions, lattices, tx_init=0.0), resid)
tx_free, c0_v, c1_v, rc_v = run_lm(
    make_spec(eulers, positions, lattices, tx_init=0.0, free_pose=True), resid)

print(f'{"grain pose":<14s}  {"tx recovered":>13s}  {"error":>10s}  {"cost drop":>22s}')
print(f'{"FROZEN":<14s}  {tx_fixed:>13.6f}  {abs(tx_fixed - TX_TRUE):>10.2e}  '
      f'{c0_f:>9.2e} -> {c1_f:>9.2e}')
print(f'{"FREE":<14s}  {tx_free:>13.6f}  {abs(tx_free - TX_TRUE):>10.2e}  '
      f'{c0_v:>9.2e} -> {c1_v:>9.2e}')
print(f'\ntrue tx = {TX_TRUE}')


grain pose       tx recovered       error               cost drop
FROZEN               0.400000    3.33e-16   3.27e+08 ->  1.24e-19
FREE                 0.002214    3.98e-01   3.27e+08 ->  3.23e+08

true tx = 0.4


### What the two probes together say

The exact-degeneracy story is **wrong**, and the measurement says so: counter-
rotating the grains does not absorb a `tx` offset, it makes the fit worse. `tx`
moves `η` only — `2θ` and `ω` are invariant under it — whereas rotating a grain
about the beam also changes the `ω` at which each of its spots diffracts. That
`ω` mismatch is exactly the *multi-grain ω-coupling* that makes `tx`
identifiable in the first place.

But probe 2 shows the practical answer is unchanged: with the pose thawed the
refinement **fails anyway** — it stalls near the start, barely moving the cost
and leaving `tx` at ~0 instead of 0.4°. Near-degenerate directions plus 24
weakly-determined nuisance parameters wreck the conditioning, even though the
problem is formally identifiable.

So: freeze the pose. Not because `tx` is unidentifiable with it free, but
because the frozen-pose problem is a one-parameter fit with a clean minimum
(the scan above) and the thawed one is not. Pose and strain get refined
downstream in `process_grains`, where they belong.


## 3. Recovering a known `tx` from zero

The real use case: the powder pass handed us `tx = 0`, and we want the truth
back. Both η-sensitive losses work — `angular` (the 3-D `2θ, η, ω` residual,
the default) and `internal_angle` (the angle between predicted and observed
g-vectors). `pixel` is rejected by the API for the reason measured in part 1.


In [7]:
for kind in ('angular', 'internal_angle'):
    r = make_residual(model, obs, matches, raw_yz, fixed_geo=fg, kind=kind)
    tx_rec, c0, c1, rc = run_lm(make_spec(eulers, positions, lattices, tx_init=0.0), r)
    print(f'{kind:<16s}  tx = {tx_rec:+.6f}  (true {TX_TRUE:+.3f}, '
          f'err {abs(tx_rec - TX_TRUE):.2e})  cost {c0:.2e} -> {c1:.2e}  rc={rc}')


angular           tx = +0.400000  (true +0.400, err 3.33e-16)  cost 3.27e+08 -> 1.24e-19  rc=0


internal_angle    tx = +0.400000  (true +0.400, err 3.33e-16)  cost 3.27e+08 -> 1.24e-19  rc=0


## 4. The production path — a real reconstruction

Everything above is the *why*. On real data you call one function (or one CLI
command) against a finished FF reconstruction. It reads `Grains.csv`,
`SpotMatrix.csv` and `hkls.csv` from the layer directory, picks the
best-**fitting** grains (smallest mean g-vector angle at the seed pose — not the
highest confidence, which admits badly-fit grains whose residuals swamp `tx`'s
sub-degree signal), fits one shared `tx` across all of them, and writes a
corrected parameter file.

> **The trap.** Pass the **master** FF parameter file — the one
> `build_paramstest` / `FitSetupParams` wrote, carrying `OmegaStep`,
> `NrFilesPerSweep`, `NrPixelsY/Z`. Do **not** pass the stripped per-layer
> `paramstest.txt` that the refiner consumes: it drops the ω-scan and detector
> keys, `OmegaStep` silently defaults to 0, every predicted spot is invalid, and
> you get `matched spots = 0` and `tx = 0`. There is a guard that raises on
> this now, but the symptom is otherwise baffling.

Set the two paths below to run it. The cell no-ops if they are not set, which
is how it stays in the automated notebook sweep.


In [8]:
from pathlib import Path

# ---- point these at a finished FF reconstruction -------------------------
LAYER_DIR    = None      # e.g. Path('/data/ni_recon/LayerNr_1')
MASTER_PARAM = None      # e.g. Path('/data/nb_ps_ni_v2.txt')  <-- MASTER, not the layer's
# -------------------------------------------------------------------------

if LAYER_DIR is None or MASTER_PARAM is None:
    print('No dataset configured - skipping the real-data step.')
    print('Set LAYER_DIR and MASTER_PARAM above to run it.')
else:
    from midas_joint_ff_calibrate.grain_refine import refine_geometry_from_grains
    out_ps = Path(LAYER_DIR) / 'paramstest_graintx.txt'
    res = refine_geometry_from_grains(
        paramstest=MASTER_PARAM, layer_dir=LAYER_DIR,
        refine_params=('tx',),          # add 'Wedge' to co-refine the rotation axis
        kind='angular', max_grains=50, max_iter=50,
        out_paramstest=out_ps, device='cpu')
    print(f'grains={res.n_grains}  matched spots={res.n_spots_matched}  rc={res.rc}')
    print(f'cost: {res.cost_init:.4e} -> {res.cost_final:.4e}')
    for k, v in res.refined.items():
        print(f'  {k} = {v:+.6f} deg')
    print(f'wrote {res.paramstest_out}')


No dataset configured - skipping the real-data step.
Set LAYER_DIR and MASTER_PARAM above to run it.


### The same thing from a terminal

```bash
midas-joint-ff-calibrate grain-tx \
    --paramstest /data/nb_ps_ni_v2.txt \
    --layer-dir  /data/ni_recon/LayerNr_1 \
    --refine tx --kind angular --max-grains 50 \
    --out /data/ni_recon/LayerNr_1/paramstest_graintx.txt
```

Or as an optional pipeline stage, which runs it automatically after
`process_grains` and writes the corrected parameter file for you:

```bash
midas-pipeline run --scan-mode ff ... --grain-geometry-run --grain-geometry-refine tx
```

## Completing the second pass

`grain-tx` writes a corrected copy of the parameter file you gave it. To
actually benefit, fold the refined `tx` into your **master** parameter file and
re-run the pipeline from the start — `tx` enters at the *transforms* step, so
indexing and refinement both have to be redone:

```python
import re
tx_ref = res.refined['tx']
txt = MASTER_PARAM.read_text()
txt = (re.sub(r'(?m)^tx\b.*$', f'tx {tx_ref:.6f}', txt)
       if re.search(r'(?m)^tx\b', txt) else txt + f'\ntx {tx_ref:.6f}\n')
MASTER_PARAM.with_name(MASTER_PARAM.stem + '_tx.txt').write_text(txt)
```

Then compare pass 1 against pass 2 — grain count, mean confidence, and the
strain spread should all improve. A worked end-to-end example (raw CBF →
calibrate → reconstruct → `grain-tx` → re-reconstruct) is
`midas_pipeline` notebook
`06_ff_cbf_real_data_calibrate_and_reconstruct.ipynb`, section 5.

## Limitations

* **Only `tx` and `Wedge` can be refined here.** `refine_geometry_from_grains`
  builds its spec with those two; `Lsd`, `BC_y`, `BC_z`, `ty`, `tz` and the
  distortion coefficients are passed as fixed geometry, so asking for them
  raises `KeyError`. The archived C `FitGrain` fit ten geometry parameters —
  that breadth has not been ported, because `tx` is the one powder genuinely
  cannot see.
* **`tx` needs several grains.** One grain cannot separate `tx` from its own
  orientation; the ω-coupling across differently-oriented grains is what breaks
  the degeneracy. The default `max_grains=50` is a reasonable floor.
* **`tx` is η-only.** It leaves `2θ` and `ω` untouched, so it does not fix a
  wrong `Lsd` or a hydrostatic strain offset. For those, see the `d0` recovery
  in `midas_pipeline` notebook 06 section 6.
